# Day 18: Memory Systems for Agents

Walkthrough of short-term, long-term, and episodic memory, retrieval with a budget, update rules, staleness, and an agent that remembers across sessions.

## 1. What is agent memory

Short-term holds the current conversation. Long-term holds stable facts saved to disk. Episodic holds a log of what actually happened.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('../day-15-agentic-ai'))
sys.path.insert(0, os.path.abspath('../day-16-custom-tools'))
print("Short-term: this conversation. Long-term: stable facts. Episodic: a log of events.")
print("A good receptionist remembers your name, your usual order, and what you talked about last time.")

## 2. Short-term memory

A bounded window of the most recent messages, oldest dropped automatically.

In [ ]:
from short_term_memory import ShortTermMemory

memory = ShortTermMemory(max_messages=4)
for role, content in [
    ("user", "What is RAG?"),
    ("agent", "Retrieval-Augmented Generation."),
    ("user", "Does it need a vector database?"),
    ("agent", "Not strictly, but it usually helps."),
    ("user", "What about caching?"),
    ("agent", "Caching can skip repeated work entirely."),
]:
    memory.add(role, content)

print(f"Messages retained: {len(memory)} out of 6 sent\n")
print(memory.as_text())

## 3. Long-term memory across sessions

Facts saved to a JSON file, loaded by a brand new instance, so a fact learned once is available later without being retold.

In [ ]:
from long_term_memory import LongTermMemory, DEFAULT_PATH

if os.path.exists(DEFAULT_PATH):
    os.remove(DEFAULT_PATH)

session_one = LongTermMemory()
session_one.remember("preferred_contact_method", "email")

session_two = LongTermMemory()
print(f"Recalled in a brand new instance: {session_two.recall('preferred_contact_method')}")

os.remove(DEFAULT_PATH)

## 4. Episodic memory

A timestamped log of events, so an agent can answer 'what happened last time' for a specific order or situation.

In [ ]:
from episodic_memory import EpisodicMemory

episodes = EpisodicMemory()
episodes.record("refund_requested", {"order_id": "ORD-1001", "amount": 25.00})
episodes.record("refund_requested", {"order_id": "ORD-1002", "amount": 89.00})
episodes.record("refund_requested", {"order_id": "ORD-1001", "note": "customer followed up"})

matches = episodes.find_by_detail("refund_requested", "order_id", "ORD-1001")
print(f"Found {len(matches)} episode(s) for ORD-1001. Most recent: {matches[0]['details']}")

## 5. Memory retrieval and budget

Ranking candidate memories by relevance to the current task, then keeping only a small budget, the same shape as document retrieval.

In [ ]:
from memory_retrieval import rank_and_limit_memories

candidates = [
    "preferred_contact_method: email",
    "favorite_product: Wireless Mouse",
    "refund_requested: order ORD-1001",
    "refund_requested: order ORD-1002",
    "ticket_created: late delivery",
]

task = "Help the customer with a refund for their order"
ranked = rank_and_limit_memories(task, candidates, budget=2)
for memory_text, score in ranked:
    print(f"score={score:.2f}  {memory_text}")

## 6. Update rules and privacy

Saving only stable facts, overwriting on correction, refusing sensitive data, and forgetting on request.

In [ ]:
from memory_update_rules import update_fact, forget_fact

facts = {}
print(update_fact(facts, "preferred_contact_method", "email"))
print(update_fact(facts, "preferred_contact_method", "phone"))
print(update_fact(facts, "payment_note", "card number 4111 1111 1111 1111"))
print(forget_fact(facts, "preferred_contact_method"))
print(f"Final facts: {facts}")

## 7. Agent with memory across two sessions

The Day 15 agent loop shape: load relevant memory before the task, record what happened after. Session two uses a fact it was never told, because session one saved it.

In [ ]:
from agent_with_memory import run_session, fake_llm_session_one, fake_llm_session_two, FACTS_PATH, EPISODES_PATH

for path in (FACTS_PATH, EPISODES_PATH):
    if os.path.exists(path):
        os.remove(path)

run_session("Session 1", fake_llm_session_one, "User mentions they'd like to be contacted by email.")
run_session("Session 2", fake_llm_session_two, "User asks to be notified when a refund is processed.")

for path in (FACTS_PATH, EPISODES_PATH):
    if os.path.exists(path):
        os.remove(path)

## 8. Staleness

A fact is only useful while it's still true. Facts older than a threshold get confirmed before being relied on, and confirming refreshes the timestamp.

In [ ]:
from memory_staleness import TimestampedFact, get_fact_for_use

fact = TimestampedFact(value="email", saved_at_tick=0)
print(f"At tick 45: is_stale = {fact.is_stale(45)}")

value, status = get_fact_for_use(fact, current_tick=45, confirm_fn=lambda v: True)
print(f"After confirming: value={value}, status={status}")
print(f"At tick 46: is_stale = {fact.is_stale(46)}")

## 9. Key takeaways

- Short-term memory holds the current conversation and stays small on purpose, so it keeps fitting in a prompt and stays focused on what's recent.
- Long-term memory holds stable facts, persisted to disk, so a fact learned once survives into a completely new session.
- Episodic memory holds a timestamped log of events, answering "what happened last time" rather than "what is true right now."
- Retrieving memory works like document retrieval: rank by relevance, apply a budget, and leave out what didn't make the cut.
- Not everything deserves to be saved: stable facts yes, one-off chatter no, sensitive data never, and a correction should overwrite rather than duplicate.
- A fact can go stale. Confirming it before relying on it, and refreshing its timestamp when confirmed, keeps memory useful without re-asking constantly.
- None of this changes the Day 15 agent loop. It only adds a load-before, save-after step around it, so the agent carries something forward between tasks instead of starting from zero every time.